# Tipos de variables

Un modelo aprende de una tabla. Cada fila es un ejemplo; una columna, el **objetivo** (*target*), es lo que el modelo debe aprender a predecir, y las demás son sus **variables** (*features*). Antes de decidir cómo preparar una columna, la primera pregunta es qué tipo de valor contiene: una cantidad, una etiqueta, un momento en el tiempo, un fragmento de texto, un nombre para la fila. Cada tipo necesita una preparación distinta, y el tipo que pandas da a una columna no responde a la pregunta.

## Una tabla pequeña

Seis coches a la venta, con una columna de cada tipo. El objetivo es el precio:

In [ ]:
import pandas as pd

cars = pd.DataFrame({
    "car_id": [1041, 1042, 1043, 1044, 1045, 1046],
    "registered": pd.to_datetime(["2016-03-14", "2019-07-01", "2012-11-23", "2021-02-09", "2018-05-30", "2015-09-17"]),
    "km": [98_500, 41_200, 187_000, 12_900, 63_300, 120_750],
    "doors": [5, 3, 5, 5, 3, 5],
    "fuel": ["diesel", "petrol", "diesel", "electric", "petrol", "hybrid"],
    "condition": ["good", "excellent", "fair", "excellent", "good", "poor"],
    "automatic": [False, False, False, True, True, False],
    "postcode": [15701, 36201, 15001, 27002, 32003, 15701],
    "description": [
        "One owner, full service history",
        "City car, new tyres",
        "High mileage, runs well",
        "Under warranty, home charger included",
        "Small dent on the rear door",
        "Needs a new clutch",
    ],
    "price": [9_800, 11_500, 3_900, 24_900, 14_200, 4_500],
})
cars

,car_id,registered,km,doors,fuel,condition,automatic,postcode,description,price
0,1041,2016-03-14,98500,5,diesel,good,False,15701,"One owner, full service history",9800
1,1042,2019-07-01,41200,3,petrol,excellent,False,36201,"City car, new tyres",11500
2,1043,2012-11-23,187000,5,diesel,fair,False,15001,"High mileage, runs well",3900
3,1044,2021-02-09,12900,5,electric,excellent,True,27002,"Under warranty, home charger included",24900
4,1045,2018-05-30,63300,3,petrol,good,True,32003,Small dent on the rear door,14200
5,1046,2015-09-17,120750,5,hybrid,poor,False,15701,Needs a new clutch,4500


Cada columna es de uno de estos tipos:

| Tipo | En la tabla | Qué es |
| --- | --- | --- |
| **Número continuo** | `km`, `price` | Una cantidad que puede tomar cualquier valor dentro de un rango. Las diferencias significan algo: 10 000 km más son 10 000 km más, sea cual sea el punto de partida |
| **Número discreto** | `doors` | Un recuento: números enteros, pero aun así una cantidad, con orden y diferencias con sentido |
| **Categoría nominal** | `fuel` | Una etiqueta de un conjunto fijo, sin orden: el diésel no es más ni menos que la gasolina |
| **Categoría ordinal** | `condition` | Una etiqueta con orden (poor < fair < good < excellent) pero sin distancias: nada dice que el salto de poor a fair sea igual al de good a excellent |
| **Booleano** | `automatic` | Una categoría con dos valores, ya un 0 o un 1 |
| **Fecha u hora** | `registered` | Un momento. Un modelo no puede usarlo tal cual, pero contiene variables: la antigüedad del coche, el mes, el día de la semana |
| **Texto libre** | `description` | Palabras, distintas en cada fila |
| **Identificador** | `car_id` | Un nombre para la fila. Dice qué coche, no nada sobre él |
| **Código escrito con dígitos** | `postcode` | Una categoría cuyas etiquetas resultan ser números |

## El tipo de dato no es la clase de variable

pandas elige un tipo para cada columna según el aspecto de los valores, no según lo que significan:

In [ ]:
cars.dtypes

car_id                  int64
registered     datetime64[us]
km                      int64
doors                   int64
fuel                      str
condition                 str
automatic                bool
postcode                int64
description               str
price                   int64
dtype: object

`car_id` y `postcode` se guardan como enteros, igual que `km` y `doors`. Cualquier cosa que trate los enteros como cantidades tratará también estas dos como cantidades. `describe` las resume junto a los números de verdad:

In [ ]:
cars.describe().loc[["mean", "min", "max"], ["km", "doors", "postcode", "car_id"]]

,km,doors,postcode,car_id
mean,87275.0,4.333333,23601.5,1043.5
min,12900.0,3.000000,15001.0,1041.0
max,187000.0,5.000000,36201.0,1046.0


La media de los códigos postales es un número que no significa nada: el código postal 15701 no está «entre» 15001 y 27002 en ningún sentido útil, y un modelo que recibiera esta columna como número aprendería justo ese tipo de relación sin sentido, sin ningún error ni aviso. Lo mismo pasa con `car_id`: como número permitiría a un modelo buscar una relación entre el identificador de un coche y su precio, y cualquiera que encontrara sería casualidad, o un rastro de cómo se asignaron los identificadores (por orden de llegada, por ejemplo), sin nada que decir sobre el siguiente coche.

Las columnas de texto tienen el problema contrario: `fuel`, `condition` y `description` tienen todas el tipo `str`, aunque la primera es una categoría sin orden, la segunda una categoría con orden y la tercera texto libre.

La clase se puede declarar. Un tipo categórico con un orden explícito hace que `condition` se ordene por su significado en lugar de alfabéticamente:

In [ ]:
condition_order = pd.CategoricalDtype(["poor", "fair", "good", "excellent"], ordered=True)

pd.DataFrame({
    "as text": cars["condition"].sort_values().to_list(),
    "as an ordered category": cars["condition"].astype(condition_order).sort_values().to_list(),
})

,as text,as an ordered category
0,excellent,poor
1,excellent,fair
2,fair,good
3,good,good
4,good,excellent
5,poor,excellent


Ordenado como texto, "excellent" va primero y "poor" el último, por sus primeras letras. Declarada como categoría ordenada, la columna se ordena de peor a mejor, y comparaciones como `>= "good"` siguen el orden. Declarar las clases pronto, al leer los datos, las mantiene en un solo sitio para pandas, pero los codificadores de scikit-learn no leen el orden declarado ahí: `OrdinalEncoder` sigue ordenando las categorías alfabéticamente salvo que se le vuelva a dar el orden. [Codificar categorías](../categorical_encoding.ipynb) convierte en números los dos tipos de categoría.

Desde pandas 3, las columnas de texto tienen el tipo `str`; antes tenían el tipo genérico `object`. El código escrito para versiones anteriores suele seleccionar las columnas de texto con `select_dtypes(include="object")`. pandas 3 todavía lo acepta, con un aviso de que dejará de hacerlo: selecciónalas con `include="str"`.

## Una tabla real: los pasajeros del Titanic

La lista de pasajeros del Titanic, que se hundió en 1912, es una de las tablas más usadas para enseñar, y con razón: contiene casi todos los tipos de columna. OpenML, un repositorio abierto de datasets para aprendizaje automático, la publica; scikit-learn la descarga la primera vez y guarda una copia en `data/`:

In [ ]:
from sklearn.datasets import fetch_openml

titanic = fetch_openml("titanic", version=1, as_frame=True, data_home="data")
passengers = titanic.frame
passengers.head()

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


El objetivo es `survived`. Un repaso de las demás columnas, con su tipo, el número de valores distintos y el número de ausentes, es la forma más rápida de empezar a clasificarlas:

In [ ]:
pd.DataFrame({
    "type": passengers.dtypes.astype(str),
    "distinct values": passengers.nunique(),
    "missing": passengers.isna().sum(),
})

,type,distinct values,missing
pclass,int64,3,0
survived,category,2,0
name,str,1307,0
sex,category,2,0
age,float64,98,263
sibsp,int64,7,0
parch,int64,8,0
ticket,str,929,0
fare,float64,281,1
cabin,str,186,1014


Leyendo cada columna con el repaso y las descripciones de las variables del dataset original (*titanic3*, recopilado por Thomas Cason a partir de la Encyclopedia Titanica; `titanic.DESCR` da su fuente):

| Columna | Tipo de dato | Clase |
| --- | --- | --- |
| `pclass` | entero | Categoría ordinal: primera, segunda o tercera clase, «un indicador aproximado de la clase socioeconómica», como dice la descripción. Tratarla como número supone que el salto de primera a segunda clase es igual al de segunda a tercera |
| `name` | texto | Texto libre, casi un valor por pasajero (1307 para 1309 filas), así que inútil tal cual; pero contiene un tratamiento (Mr, Mrs, Miss, Master) que es una categoría, una variable que extraer ([Crear variables](creating_features.ipynb)) |
| `sex`, `embarked` | categoría | Categorías nominales (`embarked` es el puerto: Cherburgo, Queenstown o Southampton) |
| `age`, `fare` | float | Números continuos. `age` falta en 263 pasajeros ([Valores ausentes](../missing_values.ipynb)) |
| `sibsp`, `parch` | entero | Números discretos: hermanos o cónyuges, y padres o hijos, que viajaban con el pasajero |
| `ticket` | texto | Un código: 929 valores distintos, compartidos solo por quienes viajaban juntos |
| `cabin` | texto | Un código cuya primera letra es la cubierta; falta en 1014 pasajeros |
| `boat`, `body` | texto, float | El bote salvavidas, y el número asignado a un cuerpo recuperado. **Ambos solo se conocen después del hundimiento** |
| `home.dest` | texto | Texto libre: dónde vivía el pasajero y adónde iba |

## Columnas que no deben ser variables

Algunas columnas deben dejarse fuera sea cual sea su tipo. Las más claras son las que contienen la respuesta. `boat` frente a `survived`:

In [ ]:
pd.crosstab(passengers["boat"].notna(), passengers["survived"])

survived,0,1
boat,,
False,800,23
True,9,477


De los 486 pasajeros con un bote salvavidas registrado, sobrevivieron 477; de los 823 sin él, 23. Un modelo que recibiera `boat` parecería casi perfecto y sería inútil: cuando haría falta la predicción, antes de que nadie suba a un bote, la columna aún no existe. `body` es peor: solo existe para los pasajeros que murieron. Una variable que solo se conoce después del resultado es una fuga del objetivo ([Fuga de datos](../../supervised-learning/ml-fundamentals/data_leakage.ipynb)).

Los identificadores son el otro caso. Una columna con un valor distinto en casi cada fila, como `name` o `car_id`, permite a un modelo memorizar filas en lugar de aprender algo que se aplique a filas nuevas. `ticket` y `cabin` son códigos a medio camino: tal cual, identifican grupos de pasajeros, pero partes de ellos (la cubierta en `cabin`) pueden ser variables de verdad.

## Más allá de las tablas

No todos los datasets son una tabla de columnas de distintos tipos:

- Las **imágenes** son arrays de números de un solo tipo: cada píxel es un brillo (gris) o tres (rojo, verde y azul), normalmente un número entero de 0 a 255. [Matrices y tensores en IA](https://github.com/avidaldo/compare-python-c-efficiency/blob/main/02_matrices_in_ai.ipynb) muestra cómo se guardan; [Escalar imágenes](scaling_images.ipynb) explica por qué se escalan de forma distinta a una tabla.
- El **texto** tiene que convertirse en números antes de que un modelo pueda usarlo: contando las palabras de cada documento, o como *embeddings*, vectores que colocan cerca los textos con significados parecidos. Modelos de lenguaje *(próximamente)* parte del texto como datos.

Dentro de una tabla, una columna de texto libre como `description` es el mismo problema en pequeño.

## Qué necesita cada tipo

| Tipo | Preparación habitual | Dónde |
| --- | --- | --- |
| Número continuo | Nada para los árboles; escalado para los modelos basados en distancias, gradientes o penalizaciones; una transformación si su distribución tiene una cola larga | [Escalar variables numéricas](../feature_scaling.ipynb), [Distribuciones asimétricas](skewed_distributions.ipynb) |
| Número discreto | Como un número continuo | |
| Categoría nominal | Codificación *one-hot*, o codificación por objetivo cuando hay muchas categorías | [Codificar categorías](../categorical_encoding.ipynb) |
| Categoría ordinal | Codificación ordinal, con el orden dado explícitamente | [Codificar categorías](../categorical_encoding.ipynb) |
| Booleano | Nada: ya es un 0 o un 1 | |
| Fecha u hora | Extraer números de ella: antigüedad, mes, día de la semana, tiempo transcurrido | [Crear variables](creating_features.ipynb) |
| Texto libre | Extraer variables (un tratamiento, una longitud) o representar todo el texto como números | [Crear variables](creating_features.ipynb), Modelos de lenguaje *(próximamente)* |
| Identificador | Quitarlo de las variables (conservándolo para enlazar las filas con su origen) | |
| Cualquier columna que solo se conoce después del resultado | Quitarla | [Fuga de datos](../../supervised-learning/ml-fundamentals/data_leakage.ipynb) |

Los valores ausentes y los valores extremos pueden aparecer en una columna de cualquier tipo: [Valores ausentes](../missing_values.ipynb) y [Valores atípicos y valores topados](outliers.ipynb).